# 00 — Environment Setup & Cache Building

**Run this notebook once before any other notebook.**

Builds all HDF5 caches required by notebooks 01–04.
All caches are stored in `processed_data/` at the repo root.

| Cache | File | Shape | Pipeline | Build time |
|-------|------|-------|----------|-----------|
| FV baseline | `processed_data/fv_original.h5` | (134, 505) | Fixed (normalize → detect) | ~2 min |
| FV windowed | `processed_data/fv_windowed.h5` | (1340, 505) | Fixed | **~40 min** |
| DGT baseline | `processed_data/dgt_original.h5` | (134, 150, 150) | Fixed | ~1 min |
| DGT windowed | `processed_data/dgt_windowed.h5` | (~1340, 150, 150) | Fixed | ~5 min |
| Original (nb01) | `processed_data/ML_data/processed_data.h5` | (146, 505) | Original (no normalize) | ~2 min |

**Prerequisites:** Raw `.bin` transient files in `original_dataset/` (not versioned, ~1.6 GB).

## 1. Feature-vector caches (RF-DNA, 505-dim)

In [1]:
import sys, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from dl_classification.data_loader_bridge import RFPreprocessor

# Baseline: one 505-dim feature vector per transient (~2 min)
RFPreprocessor(windowed=False).load()

# Windowed: 10 overlapping windows per transient (~40 min from raw .bin files)
RFPreprocessor(windowed=True, window_size=300, window_step=300,
               max_windows_per_transient=10).load()

[RFPreprocessor] Loading cache: /Users/matteocampagnaro/Documents/Advanced-Security/processed_data/fv_original.h5
[RFPreprocessor] 134 samples | 12 classes | windowed=False
[RFPreprocessor] Loading cache: /Users/matteocampagnaro/Documents/Advanced-Security/processed_data/fv_windowed.h5
[RFPreprocessor] 1340 samples | 12 classes | windowed=True


(array([[1.1003847e-01, 1.2108465e-02, 2.6736867e-01, ..., 2.8254457e+00,
         9.8880606e+00, 4.0580192e+00],
        [1.6180144e-01, 2.6179707e-02, 4.1622970e-01, ..., 2.6148067e+00,
         8.7095089e+00, 3.9077332e+00],
        [2.6981637e-01, 7.2800867e-02, 1.2473487e+00, ..., 4.1009493e+00,
         2.1989647e+01, 4.0295348e+00],
        ...,
        [2.8043979e-01, 7.8646481e-02, 1.7918307e+00, ..., 4.1431022e+00,
         2.2844404e+01, 4.1104469e+00],
        [1.3937107e-01, 1.9424295e-02, 1.3873880e+00, ..., 3.5913274e+00,
         1.6795559e+01, 4.0330005e+00],
        [8.1286877e-02, 6.6075562e-03, 1.0219485e+00, ..., 4.0417504e+00,
         2.3688749e+01, 4.0607343e+00]], dtype=float32),
 array([ 0,  0,  0, ..., 11, 11, 11]),
 {'device1': 0,
  'device2': 1,
  'device3': 2,
  'device4': 3,
  'device5': 4,
  'device6': 5,
  'device7': 6,
  'device8': 7,
  'device9': 8,
  'device10': 9,
  'device11': 10,
  'device12': 11},
 ['device1',
  'device2',
  'device3',
  'device4

## 2. DGT matrix caches (150×150, `max_windows=10`)

> **Important:** Pass `max_windows=10` explicitly. The default is 3, which would produce only ~390 samples and a stale `windows_per_transient` attribute — causing wrong transient grouping in the train/val/test split.

In [2]:
from raw_dgt.dgt_data_loader import DGTPreprocessor

DGTPreprocessor(windowed=False).load()
DGTPreprocessor(windowed=True, max_windows=10).load(force=True)

[DGTPreprocessor] Loading cache: /Users/matteocampagnaro/Documents/Advanced-Security/processed_data/dgt_original.h5
  134 samples, shape (134, 150, 150)

[DGTPreprocessor | windowed] 12 devices
  window=300  step=300  cap=10
------------------------------------------------------------
  device1: 130 matrices
  device2: 130 matrices
  device3: 130 matrices
  device4: 120 matrices
  device5: 130 matrices
  device6: 130 matrices
  device7: 130 matrices
  device8: 130 matrices
  device9: 130 matrices
  device10: 60 matrices
  device11: 60 matrices
  device12: 60 matrices

[DGTPreprocessor | windowed] 1340 total  (6s)
  Saved → /Users/matteocampagnaro/Documents/Advanced-Security/processed_data/dgt_windowed.h5


(array([[[0.22485918, 0.12798297, 0.0249752 , ..., 0.16121823,
          0.19920546, 0.21283267],
         [0.22159533, 0.12267049, 0.01622276, ..., 0.17556518,
          0.20125614, 0.20783617],
         [0.21779042, 0.11812197, 0.00931428, ..., 0.1896262 ,
          0.20230904, 0.20152594],
         ...,
         [0.05714819, 0.0859394 , 0.18892442, ..., 0.6057491 ,
          0.30029848, 0.11079624],
         [0.05795384, 0.08707745, 0.19333805, ..., 0.58909637,
          0.298482  , 0.1126212 ],
         [0.0580145 , 0.08750076, 0.19744608, ..., 0.5706423 ,
          0.2954077 , 0.11360099]],
 
        [[0.42969054, 0.06153346, 0.01565398, ..., 0.093224  ,
          0.34233448, 0.606018  ],
         [0.41078547, 0.05392672, 0.01281214, ..., 0.08404496,
          0.3239008 , 0.5818619 ],
         [0.3934223 , 0.04839995, 0.01085936, ..., 0.07556712,
          0.30582997, 0.55796576],
         ...,
         [0.5539757 , 0.82880694, 0.8218233 , ..., 0.13368694,
          0.02263765, 0.

## 3. Original pipeline cache (for notebook 01)

Notebook 01 replicates `RF_Fingerprint.py` faithfully, including a bug in
`src/dataloader.py` where `normalize_magnitude()` was commented out before
`detect_transients()`. This produces **146 samples** (vs 134 from the fixed
pipeline above) and is the dataset needed to reproduce the reported ~0.97 ADR.

We replicate that exact behavior here but swap `generate_rf_dna_fingerprint()`
for `fast_rf_dna_fingerprint()` (numpy-FFT-based, ~50× faster, numerically
identical for N=1) to keep build time under 2 minutes instead of ~45 minutes.

In [3]:
import re, time
import numpy as np
import h5py

from src.preprocessing import (
    load_iq_data, detect_transients, filter_transients, apply_lowpass_filter,
)
from dl_classification.fast_fingerprint import fast_rf_dna_fingerprint

ORIG_CACHE_PATH = os.path.join(ROOT, 'processed_data', 'ML_data', 'processed_data.h5')
RAW_DATA_DIR    = os.path.join(ROOT, 'original_dataset')

_ORIG_PARAMS = dict(
    sample_rate=20e6, cutoff_freq=5e6, M=150, KG=150, N=1,
    NP=100, NT=15, NF=15,
    transient_threshold=0.38,
    specific_duration_threshold=0.005,
    specific_magnitude_threshold=0.3,
    min_transient_duration=0.005,
    filter_type='chebyshev', filter_order=4, filter_ripple=0.5,
    mode='diagonal',
)

def _natural_sort_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'([0-9]+)', s)]

if os.path.exists(ORIG_CACHE_PATH):
    print(f'Original pipeline cache already exists:\n  {ORIG_CACHE_PATH}')
else:
    print(f'Building original pipeline cache → {ORIG_CACHE_PATH}')
    p = _ORIG_PARAMS

    device_folders = sorted(
        [f for f in os.listdir(RAW_DATA_DIR)
         if os.path.isdir(os.path.join(RAW_DATA_DIR, f))],
        key=_natural_sort_key,
    )
    device_id_mapping = {name: idx for idx, name in enumerate(device_folders)}
    total_devices     = list(device_id_mapping.keys())

    all_data, all_labels = [], []
    t0 = time.time()

    for device_name, device_id in device_id_mapping.items():
        folder_path = os.path.join(RAW_DATA_DIR, device_name)
        bin_files   = sorted(
            [f for f in os.listdir(folder_path) if f.endswith('.bin')],
            key=_natural_sort_key,
        )
        dev_count = 0
        for fname in bin_files:
            try:
                iq_data = load_iq_data(os.path.join(folder_path, fname), 0, -1)
                # intentionally NO normalize_magnitude here — replicates the original
                # src/dataloader.py where that call was commented out
                starts, ends = detect_transients(
                    iq_data, p['sample_rate'],
                    p['transient_threshold'], p['min_transient_duration'],
                )
                selected = filter_transients(
                    starts, ends, iq_data, p['sample_rate'],
                    p['specific_duration_threshold'], p['specific_magnitude_threshold'],
                )
                for start, end in selected:
                    if end <= start:
                        continue
                    filtered = apply_lowpass_filter(
                        iq_data[start:end], p['cutoff_freq'], p['sample_rate'],
                        filter_type=p['filter_type'],
                        order=p['filter_order'],
                        ripple=p['filter_ripple'],
                    )
                    fp = fast_rf_dna_fingerprint(
                        filtered, fs=p['sample_rate'],
                        M=p['M'], KG=p['KG'], N=p['N'],
                        NP=p['NP'], NT=p['NT'], NF=p['NF'], mode=p['mode'],
                    )
                    all_data.append(fp)
                    all_labels.append(device_id)
                    dev_count += 1
            except Exception as e:
                print(f'    Skipping {fname}: {e}')
        print(f'  {device_name}: {dev_count} fingerprints')

    X_orig = np.array(all_data,   dtype=np.float32)
    y_orig = np.array(all_labels, dtype=np.int64)

    os.makedirs(os.path.dirname(ORIG_CACHE_PATH), exist_ok=True)
    with h5py.File(ORIG_CACHE_PATH, 'w') as f:
        f.create_dataset('data',   data=X_orig)
        f.create_dataset('labels', data=y_orig)
        f.create_dataset('device_id_mapping', data=np.string_(str(device_id_mapping)))
        f.create_dataset('total_devices',     data=np.string_(str(total_devices)))

    print(f'\nDone in {(time.time()-t0)/60:.1f} min  |  X={X_orig.shape}')
    print(f'Saved → {ORIG_CACHE_PATH}')

Original pipeline cache already exists:
  /Users/matteocampagnaro/Documents/Advanced-Security/processed_data/ML_data/processed_data.h5


## 3. Validation — check shapes and attributes

In [4]:
import h5py, sys, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from binary_pla.config import FV_ORIGINAL_PATH, FV_WINDOWED_PATH, DGT_ORIGINAL_PATH, DGT_WINDOWED_PATH

caches = [
    ("FV baseline",  FV_ORIGINAL_PATH),
    ("FV windowed",  FV_WINDOWED_PATH),
    ("DGT baseline", DGT_ORIGINAL_PATH),
    ("DGT windowed", DGT_WINDOWED_PATH),
]

print(f"{'Cache':<15} {'Shape':<22} {'windows_per_transient'}")
print("-" * 55)
all_ok = True
for name, path in caches:
    if not os.path.exists(path):
        print(f"{name:<15} MISSING  ← {path}")
        all_ok = False
        continue
    with h5py.File(path, "r") as f:
        shape = f["data"].shape
        n_win = f["data"].attrs.get("windows_per_transient", "(absent)")
    print(f"{name:<15} {str(shape):<22} {n_win}")

if all_ok:
    with h5py.File(DGT_WINDOWED_PATH, "r") as f:
        n = int(f["data"].attrs.get("windows_per_transient", 0))
    assert n == 10, f"DGT windowed: windows_per_transient={n}, expected 10."
    print("\n✓ All caches present. DGT windowed has windows_per_transient=10.")
else:
    print("\n✗ Some caches are missing. Run the build cells above.")

Cache           Shape                  windows_per_transient
-------------------------------------------------------
FV baseline     (134, 505)             (absent)
FV windowed     (1340, 505)            (absent)
DGT baseline    (134, 150, 150)        1
DGT windowed    (1340, 150, 150)       10

✓ All caches present. DGT windowed has windows_per_transient=10.
